<a href="https://colab.research.google.com/github/samer-glitch/TADP-Cluster-Computing/blob/main/06_TADP_Experiment_E_NewDesign_Policy_Sensitivity_LODO_K10_5Seed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
import io
import os

print("📤 Please upload the diabetes_130US.csv file:")
uploaded = files.upload()
DATA_FILENAME  = "diabetes_130US.csv"
DATA_CACHE_DIR = "./data_cache"
os.makedirs(DATA_CACHE_DIR, exist_ok=True)

# Get the uploaded file
for filename in uploaded.keys():
    file_data = uploaded[filename]
    print(f'✅ Uploaded: {filename} ({len(file_data)} bytes)')


    # Save to cache
    cache_path = os.path.join(DATA_CACHE_DIR, DATA_FILENAME)
    with open(cache_path, 'wb') as f:
        f.write(file_data)
    print(f'✅ Saved to cache: {cache_path}')

    # Verify the file was saved
    if os.path.exists(cache_path):
        file_size = os.path.getsize(cache_path)
        print(f'✅ Verified: {cache_path} exists ({file_size} bytes)')
    else:
        print(f'❌ Error: File was not saved properly')

📤 Please upload the diabetes_130US.csv file:


Saving diabetes_130US.csv to diabetes_130US.csv
✅ Uploaded: diabetes_130US.csv (19159383 bytes)
✅ Saved to cache: ./data_cache/diabetes_130US.csv
✅ Verified: ./data_cache/diabetes_130US.csv exists (19159383 bytes)


In [3]:
# %%
# ======================================================================================
# TADP EXPERIMENT E v20.2
# FRESH-REFERENCE POLICY SENSITIVITY + FULL-POLICY LODO ABLATION
# K=10 | TADP-VR only | 5 training seeds | 4 FL rounds
# ======================================================================================
#
# PURPOSE
# -------
# Experiment E is a STANDALONE TADP-VR sensitivity/ablation study.
# It does NOT load, compare against, or reuse predictive results from Experiment A1.
#
# The unmodified final policy is first executed inside Experiment E and its TADP-VR
# reference cohort is trained FRESH over the five training seeds. Every sensitivity or
# LODO condition is then compared with this fresh within-experiment reference.
#
# E1) POLICY THRESHOLD SENSITIVITY
#     Perturb the numerical policy thresholds around the frozen reference values:
#       - HPS lower/reject threshold: 3.0
#       - HPS upper/direct-accept threshold: 3.5
#       - Review Score acceptance threshold: 3.25
#       - dimension floor: 2.5
#
#     HPS lower and upper thresholds are tested separately and jointly.
#     Review Score and dimension-floor thresholds are tested separately.
#     Perturbations: +/-10%, +/-20%, +/-30%.
#
#     Mandatory-factor minima are NOT perturbed. They encode fixed governance
#     requirements rather than tunable score thresholds.
#
# E2) FULL-POLICY LEAVE-ONE-DIMENSION-OUT (LODO)
#     For each dimension d, remove d from:
#       - HPS calculation (remaining weights are renormalized to sum to 1),
#       - dimension-floor enforcement,
#       - mandatory factors belonging to d,
#       - Review Score factors belonging to d.
#
# SAME CORE DESIGN AS MAIN EXPERIMENT A1
# --------------------------------------
#     K = 10
#     split seed = 7001
#     partition seed = 7101
#     frozen evidence seed = 1042
#     training seeds = [42, 142, 242, 342, 442]
#     rounds = 4
#     local epochs = 1
#     batch = 64
#     Dirichlet alpha = 1.0
#     expected reference admitted cohort = A, B, G, H, I, J
#
# FINAL POLICY ORDER
# ------------------
# 1) mandatory gate: any active mandatory requirement below its minimum -> REJECT
# 2) dimension floor: any active dimension < floor -> REJECT
# 3) HPS < lower threshold -> REJECT
# 4) HPS >= upper threshold -> DIRECT ACCEPT
# 5) lower <= HPS < upper -> automated Review Score
#       Review Score >= review threshold -> ACCEPT
#       otherwise -> REJECT
#
# PRIMARY COMPARISONS — ALL WITHIN THIS NEW EXPERIMENT-E RUN
# ----------------------------------------------------------
# Governance impact vs fresh reference:
#   - admitted count/rate
#   - gained clients
#   - lost clients
#   - net admission change
#   - decision flips
#   - Jaccard overlap
#
# Predictive impact vs fresh reference TADP-VR, paired by training seed:
#   - delta Accuracy
#   - delta Macro-F1
#   - delta Macro ROC-AUC (primary utility endpoint)
#   - mean +/- SD and 95% CI of paired deltas
#
# COMPUTE EFFICIENCY
# ------------------
# Training is performed once per UNIQUE admitted cohort and seed. Policy settings that
# produce the same cohort reuse the same freshly trained cohort result. They do NOT reuse
# any result from a previous experiment.
#
# INPUT
# -----
# Only the Diabetes 130-US CSV is required.
# ======================================================================================

import os
import sys
import gc
import re
import json
import math
import time
import random
import hashlib
import zipfile
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


# ======================================================================================
# 1. FROZEN EXPERIMENT-E SETTINGS — SAME CORE TRAINING DESIGN AS A1 TADP-VR
# ======================================================================================

EXPERIMENT_VERSION = (
    "TADP-E-v20.2-K10-FRESHREF-POLICY-SENSITIVITY-LODO-"
    "MANDATORYGATE-REVIEWSCORE325-5SEED-4ROUND"
)

K_SUBMISSIONS = 10
TRAINING_RUN_SEEDS = [42, 142, 242, 342, 442]
NUM_ROUNDS_FL = 4
LOCAL_EPOCHS = 1
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
DIRICHLET_ALPHA = 1.0

GLOBAL_SPLIT_SEED = 7001
CLIENT_PARTITION_SEED = 7101
FROZEN_EVIDENCE_ASSIGNMENT_SEED = 1042
DOMAIN = "healthcare"

REFERENCE_LOWER_CUT = 3.0
REFERENCE_UPPER_CUT = 3.5
REFERENCE_REVIEW_CUT = 3.25
REFERENCE_DIMENSION_FLOOR = 2.5
PERTURBATION_PCTS = [-30, -20, -10, 10, 20, 30]

DIMS = [f"dim{i}" for i in range(1, 7)]
MAX_FACTOR_SCORE = 5.0
EXPECTED_CLIENTS = list("ABCDEFGHIJ")
EXPECTED_REFERENCE_COHORT = ["A", "B", "G", "H", "I", "J"]
EXPECTED_REFERENCE_ACTIONS = {
    "A": "ACCEPT", "B": "ACCEPT", "C": "REJECT", "D": "REJECT", "E": "REJECT",
    "F": "REJECT", "G": "ACCEPT", "H": "ACCEPT", "I": "ACCEPT", "J": "ACCEPT",
}
EXPECTED_REFERENCE_PATHS = {
    "A": "HPS_REVIEW_BAND",
    "B": "DIRECT_AUTO_ACCEPT",
    "C": "DIMENSION_FLOOR",
    "D": "LOW_HPS",
    "E": "HPS_REVIEW_BAND",
    "F": "MANDATORY_GOVERNANCE_GATE",
    "G": "DIRECT_AUTO_ACCEPT",
    "H": "HPS_REVIEW_BAND",
    "I": "DIRECT_AUTO_ACCEPT",
    "J": "DIRECT_AUTO_ACCEPT",
}

HPS_WEIGHTS = {
    "dim1": 0.25,
    "dim2": 0.15,
    "dim3": 0.10,
    "dim4": 0.10,
    "dim5": 0.30,
    "dim6": 0.10,
}

DIMENSION_NAMES = {
    "dim1": "Source Reliability",
    "dim2": "Data Quality and Health",
    "dim3": "Documentation Practices",
    "dim4": "Timeliness and Refresh Rate",
    "dim5": "Regulatory and Compliance Alignment",
    "dim6": "Context and Usage Constraints",
}

FACTOR_NAMES = {
    "dim1": ["source_reputation", "data_controller", "data_objective", "data_collection_lineage"],
    "dim2": [
        "completeness", "duplication_rate", "value_validity_error_rate", "type_consistency",
        "label_integrity", "feature_distribution_consistency", "feature_category_coverage",
        "structural_constraint_integrity",
    ],
    "dim3": ["data_dictionary", "version_logs", "collection_protocol", "definition_updates"],
    "dim4": ["data_freshness", "scheduled_refresh", "retention_clarity"],
    "dim5": ["regulation_coverage", "consent_ethics", "geo_restrictions", "sensitivity_classification", "audits"],
    "dim6": ["license_terms", "ethical_reviews", "redistribution", "user_agreements"],
}
DOCUMENTARY_DIMS = ("dim1", "dim3", "dim4", "dim5", "dim6")

FACTOR_ADEQUACY_MIN = {
    "dim1": {"source_reputation": 4, "data_controller": 4, "data_objective": 4, "data_collection_lineage": 4},
    "dim2": {f: 3 for f in FACTOR_NAMES["dim2"]},
    "dim3": {"data_dictionary": 3, "version_logs": 3, "collection_protocol": 4, "definition_updates": 3},
    "dim4": {"data_freshness": 3, "scheduled_refresh": 3, "retention_clarity": 4},
    "dim5": {"regulation_coverage": 3, "consent_ethics": 3, "geo_restrictions": 3, "sensitivity_classification": 3, "audits": 4},
    "dim6": {"license_terms": 3, "ethical_reviews": 4, "redistribution": 3, "user_agreements": 4},
}

MANDATORY_MINIMA = {
    ("dim1", "data_controller"): 4.0,
    ("dim1", "data_collection_lineage"): 4.0,
    ("dim5", "regulation_coverage"): 3.0,
    ("dim5", "consent_ethics"): 3.0,
    ("dim5", "sensitivity_classification"): 3.0,
    ("dim6", "license_terms"): 3.0,
}

REVIEW_FACTORS = {
    ("dim1", "source_reputation"),
    ("dim1", "data_objective"),
    ("dim3", "data_dictionary"),
    ("dim3", "version_logs"),
    ("dim3", "collection_protocol"),
    ("dim3", "definition_updates"),
    ("dim4", "data_freshness"),
    ("dim4", "scheduled_refresh"),
    ("dim4", "retention_clarity"),
    ("dim5", "geo_restrictions"),
    ("dim5", "audits"),
    ("dim6", "ethical_reviews"),
    ("dim6", "redistribution"),
    ("dim6", "user_agreements"),
}

USE_GOOGLE_DRIVE_CHECKPOINTS = True
ESTIMATED_POWER_W = 12.0

assert abs(sum(HPS_WEIGHTS.values()) - 1.0) < 1e-12
assert sum(len(v) for v in FACTOR_NAMES.values()) == 28
assert len(MANDATORY_MINIMA) == 6
assert len(REVIEW_FACTORS) == 14


# ======================================================================================
# 2. GENERAL UTILITIES
# ======================================================================================

def print_banner(title: str, width: int = 112):
    print("\n" + "=" * width)
    print(title)
    print("=" * width)


def seed_everything(seed: int):
    seed = int(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        tf.keras.utils.set_random_seed(seed)
    except Exception:
        tf.random.set_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


def sha256_file(path: str | Path) -> str:
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def sha256_weights(weights: List[np.ndarray]) -> str:
    h = hashlib.sha256()
    for w in weights:
        a = np.ascontiguousarray(np.asarray(w))
        h.update(str(a.shape).encode())
        h.update(a.view(np.uint8).tobytes())
    return h.hexdigest()


def ensure_dir(path: str | Path) -> Path:
    p = Path(path)
    p.mkdir(parents=True, exist_ok=True)
    return p


def choose_experiment_root(experiment_name: str, use_drive: bool = True) -> Path:
    if use_drive:
        try:
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
            root = Path("/content/drive/MyDrive/TADP_CHECKPOINTS") / experiment_name
            root.mkdir(parents=True, exist_ok=True)
            print(f"Persistent checkpoint root: {root}")
            return root
        except Exception as exc:
            print(f"Google Drive checkpoint mount unavailable: {exc}")
    root = Path("/content") / experiment_name
    root.mkdir(parents=True, exist_ok=True)
    print(f"Local checkpoint root: {root}")
    return root


def atomic_write_json(obj: dict, path: str | Path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")
    tmp.replace(path)


def load_checkpoint_state(path: str | Path) -> dict:
    path = Path(path)
    if not path.exists():
        return {"completed": [], "last_completed": None, "updated_utc": None}
    return json.loads(path.read_text(encoding="utf-8"))


def mark_checkpoint_complete(state_path, key):
    from datetime import datetime, timezone
    state = load_checkpoint_state(state_path)
    completed = list(state.get("completed", []))
    if key not in completed:
        completed.append(key)
    state["completed"] = completed
    state["last_completed"] = key
    state["updated_utc"] = datetime.now(timezone.utc).isoformat()
    atomic_write_json(state, state_path)


def upsert_csv(row, path, key_cols):
    path = Path(path)
    new = pd.DataFrame([row])
    if path.exists():
        old = pd.read_csv(path)
        if not old.empty:
            mask = pd.Series(True, index=old.index)
            for c in key_cols:
                mask &= old[c].astype(str).eq(str(row[c]))
            old = old.loc[~mask].copy()
            new = pd.concat([old, new], ignore_index=True)
    new.to_csv(path, index=False)


def bool_series(s: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(s):
        return s.fillna(False)
    return (
        s.astype(str).str.strip().str.lower()
        .map({"true": True, "1": True, "yes": True, "false": False, "0": False, "no": False})
        .fillna(False).astype(bool)
    )


def ci95_mean(values) -> Tuple[float, float]:
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan
    if len(x) == 1:
        return float(x[0]), float(x[0])
    mean = float(np.mean(x))
    sd = float(np.std(x, ddof=1))
    try:
        from scipy.stats import t
        crit = float(t.ppf(0.975, df=len(x) - 1))
    except Exception:
        crit = 1.96
    half = crit * sd / math.sqrt(len(x))
    return mean - half, mean + half


def jaccard(a, b) -> float:
    a, b = set(a), set(b)
    u = a | b
    return float(len(a & b) / len(u)) if u else 1.0


# ======================================================================================
# 3. INPUT — ONLY THE DIABETES DATASET
# ======================================================================================

def locate_diabetes_csv() -> str:
    candidates = [
        os.environ.get("DIABETES_CSV", ""),
        "/content/diabetes_130US.csv",
        "/content/diabetic_data.csv",
        "./diabetes_130US.csv",
        "./diabetic_data.csv",
        "/content/drive/MyDrive/diabetes_130US.csv",
        "/content/drive/MyDrive/diabetic_data.csv",
    ]
    for p in candidates:
        if p and os.path.exists(p):
            return str(p)
    try:
        from google.colab import files as colab_files
        print("Please upload the Diabetes 130-US CSV.")
        uploaded = colab_files.upload()
        csvs = [name for name in uploaded if str(name).lower().endswith(".csv")]
        if not csvs:
            raise RuntimeError("No CSV file was uploaded.")
        return str(csvs[0])
    except ImportError:
        pass
    raise FileNotFoundError("Diabetes CSV not found. Set DIABETES_CSV or upload it in Colab.")


# ======================================================================================
# 4. FINAL GOVERNANCE DESIGN — GENERATED FRESH INSIDE EXPERIMENT E
# ======================================================================================

BUNDLE_TEMPLATES = {'E01': {'profile': 'DIRECT_STRONG',
         'variant': 0,
         'factors': {'dim1': {'source_reputation': 4.0,
                              'data_controller': 4.0,
                              'data_objective': 5.0,
                              'data_collection_lineage': 4.0},
                     'dim3': {'data_dictionary': 4.0,
                              'version_logs': 4.0,
                              'collection_protocol': 5.0,
                              'definition_updates': 4.0},
                     'dim4': {'data_freshness': 4.0, 'scheduled_refresh': 4.0, 'retention_clarity': 5.0},
                     'dim5': {'regulation_coverage': 4.0,
                              'consent_ethics': 4.0,
                              'geo_restrictions': 5.0,
                              'sensitivity_classification': 4.0,
                              'audits': 4.0},
                     'dim6': {'license_terms': 4.0,
                              'ethical_reviews': 4.0,
                              'redistribution': 5.0,
                              'user_agreements': 4.0}}},
 'E02': {'profile': 'DIRECT_STRONG',
         'variant': 1,
         'factors': {'dim1': {'source_reputation': 4.0,
                              'data_controller': 5.0,
                              'data_objective': 4.0,
                              'data_collection_lineage': 4.0},
                     'dim3': {'data_dictionary': 4.0,
                              'version_logs': 5.0,
                              'collection_protocol': 4.0,
                              'definition_updates': 4.0},
                     'dim4': {'data_freshness': 4.0, 'scheduled_refresh': 5.0, 'retention_clarity': 4.0},
                     'dim5': {'regulation_coverage': 4.0,
                              'consent_ethics': 5.0,
                              'geo_restrictions': 4.0,
                              'sensitivity_classification': 4.0,
                              'audits': 5.0},
                     'dim6': {'license_terms': 4.0,
                              'ethical_reviews': 5.0,
                              'redistribution': 4.0,
                              'user_agreements': 4.0}}},
 'E03': {'profile': 'DIRECT_STRONG',
         'variant': 2,
         'factors': {'dim1': {'source_reputation': 5.0,
                              'data_controller': 4.0,
                              'data_objective': 4.0,
                              'data_collection_lineage': 5.0},
                     'dim3': {'data_dictionary': 5.0,
                              'version_logs': 4.0,
                              'collection_protocol': 4.0,
                              'definition_updates': 5.0},
                     'dim4': {'data_freshness': 5.0, 'scheduled_refresh': 4.0, 'retention_clarity': 4.0},
                     'dim5': {'regulation_coverage': 5.0,
                              'consent_ethics': 4.0,
                              'geo_restrictions': 4.0,
                              'sensitivity_classification': 5.0,
                              'audits': 4.0},
                     'dim6': {'license_terms': 5.0,
                              'ethical_reviews': 4.0,
                              'redistribution': 4.0,
                              'user_agreements': 5.0}}},
 'E04': {'profile': 'DIRECT_STRONG',
         'variant': 3,
         'factors': {'dim1': {'source_reputation': 4.0,
                              'data_controller': 4.0,
                              'data_objective': 5.0,
                              'data_collection_lineage': 4.0},
                     'dim3': {'data_dictionary': 4.0,
                              'version_logs': 4.0,
                              'collection_protocol': 5.0,
                              'definition_updates': 4.0},
                     'dim4': {'data_freshness': 4.0, 'scheduled_refresh': 4.0, 'retention_clarity': 5.0},
                     'dim5': {'regulation_coverage': 4.0,
                              'consent_ethics': 4.0,
                              'geo_restrictions': 5.0,
                              'sensitivity_classification': 4.0,
                              'audits': 4.0},
                     'dim6': {'license_terms': 4.0,
                              'ethical_reviews': 4.0,
                              'redistribution': 5.0,
                              'user_agreements': 4.0}}},
 'E05': {'profile': 'REVIEW_RECOVERABLE',
         'variant': 0,
         'factors': {'dim1': {'source_reputation': 3.0,
                              'data_controller': 4.0,
                              'data_objective': 3.0,
                              'data_collection_lineage': 4.0},
                     'dim3': {'data_dictionary': 4.0,
                              'version_logs': 4.0,
                              'collection_protocol': 4.0,
                              'definition_updates': 4.0},
                     'dim4': {'data_freshness': 3.0, 'scheduled_refresh': 3.0, 'retention_clarity': 3.0},
                     'dim5': {'regulation_coverage': 3.0,
                              'consent_ethics': 3.0,
                              'geo_restrictions': 3.0,
                              'sensitivity_classification': 3.0,
                              'audits': 3.0},
                     'dim6': {'license_terms': 3.0,
                              'ethical_reviews': 3.0,
                              'redistribution': 3.0,
                              'user_agreements': 3.0}}},
 'E06': {'profile': 'REVIEW_RECOVERABLE',
         'variant': 1,
         'factors': {'dim1': {'source_reputation': 3.0,
                              'data_controller': 4.0,
                              'data_objective': 3.0,
                              'data_collection_lineage': 4.0},
                     'dim3': {'data_dictionary': 4.0,
                              'version_logs': 4.0,
                              'collection_protocol': 4.0,
                              'definition_updates': 4.0},
                     'dim4': {'data_freshness': 3.0, 'scheduled_refresh': 4.0, 'retention_clarity': 3.0},
                     'dim5': {'regulation_coverage': 3.0,
                              'consent_ethics': 3.0,
                              'geo_restrictions': 3.0,
                              'sensitivity_classification': 3.0,
                              'audits': 3.0},
                     'dim6': {'license_terms': 3.0,
                              'ethical_reviews': 3.0,
                              'redistribution': 3.0,
                              'user_agreements': 2.0}}},
 'E07': {'profile': 'MANDATORY_GATE_FAIL',
         'variant': 0,
         'factors': {'dim1': {'source_reputation': 4.0,
                              'data_controller': 3.0,
                              'data_objective': 4.0,
                              'data_collection_lineage': 4.0},
                     'dim3': {'data_dictionary': 4.0,
                              'version_logs': 4.0,
                              'collection_protocol': 4.0,
                              'definition_updates': 4.0},
                     'dim4': {'data_freshness': 4.0, 'scheduled_refresh': 4.0, 'retention_clarity': 4.0},
                     'dim5': {'regulation_coverage': 4.0,
                              'consent_ethics': 4.0,
                              'geo_restrictions': 4.0,
                              'sensitivity_classification': 4.0,
                              'audits': 4.0},
                     'dim6': {'license_terms': 4.0,
                              'ethical_reviews': 4.0,
                              'redistribution': 4.0,
                              'user_agreements': 4.0}}},
 'E08': {'profile': 'REVIEW_LIMITED',
         'variant': 0,
         'factors': {'dim1': {'source_reputation': 3.0,
                              'data_controller': 4.0,
                              'data_objective': 3.0,
                              'data_collection_lineage': 4.0},
                     'dim3': {'data_dictionary': 3.0,
                              'version_logs': 3.0,
                              'collection_protocol': 3.0,
                              'definition_updates': 3.0},
                     'dim4': {'data_freshness': 3.0, 'scheduled_refresh': 3.0, 'retention_clarity': 3.0},
                     'dim5': {'regulation_coverage': 3.0,
                              'consent_ethics': 3.0,
                              'geo_restrictions': 3.0,
                              'sensitivity_classification': 3.0,
                              'audits': 3.0},
                     'dim6': {'license_terms': 3.0,
                              'ethical_reviews': 3.0,
                              'redistribution': 3.0,
                              'user_agreements': 3.0}}},
 'E09': {'profile': 'LOW_HPS_WEAK',
         'variant': 0,
         'factors': {'dim1': {'source_reputation': 2.0,
                              'data_controller': 4.0,
                              'data_objective': 2.0,
                              'data_collection_lineage': 4.0},
                     'dim3': {'data_dictionary': 2.0,
                              'version_logs': 3.0,
                              'collection_protocol': 2.0,
                              'definition_updates': 3.0},
                     'dim4': {'data_freshness': 2.0, 'scheduled_refresh': 3.0, 'retention_clarity': 3.0},
                     'dim5': {'regulation_coverage': 3.0,
                              'consent_ethics': 3.0,
                              'geo_restrictions': 2.0,
                              'sensitivity_classification': 3.0,
                              'audits': 2.0},
                     'dim6': {'license_terms': 3.0,
                              'ethical_reviews': 2.0,
                              'redistribution': 2.0,
                              'user_agreements': 3.0}}},
 'E10': {'profile': 'DIMENSION_FLOOR_WEAK',
         'variant': 0,
         'factors': {'dim1': {'source_reputation': 3.0,
                              'data_controller': 4.0,
                              'data_objective': 3.0,
                              'data_collection_lineage': 4.0},
                     'dim3': {'data_dictionary': 3.0,
                              'version_logs': 3.0,
                              'collection_protocol': 3.0,
                              'definition_updates': 3.0},
                     'dim4': {'data_freshness': 2.0, 'scheduled_refresh': 2.0, 'retention_clarity': 2.0},
                     'dim5': {'regulation_coverage': 3.0,
                              'consent_ethics': 3.0,
                              'geo_restrictions': 3.0,
                              'sensitivity_classification': 3.0,
                              'audits': 3.0},
                     'dim6': {'license_terms': 3.0,
                              'ethical_reviews': 3.0,
                              'redistribution': 3.0,
                              'user_agreements': 3.0}}}}


def healthcare_policy_table() -> pd.DataFrame:
    rows = []
    for dim in DIMS:
        for factor in FACTOR_NAMES[dim]:
            key = (dim, factor)
            rows.append({
                "domain": DOMAIN,
                "dimension": dim,
                "dimension_name": DIMENSION_NAMES[dim],
                "factor": factor,
                "adequacy_min_rank": float(FACTOR_ADEQUACY_MIN[dim][factor]),
                "is_mandatory_factor": bool(key in MANDATORY_MINIMA),
                "mandatory_minimum_0_5": float(MANDATORY_MINIMA[key]) if key in MANDATORY_MINIMA else np.nan,
                "is_review_factor": bool(key in REVIEW_FACTORS),
                "review_acceptance_threshold_0_5": REFERENCE_REVIEW_CUT if key in REVIEW_FACTORS else np.nan,
            })
    out = pd.DataFrame(rows)
    if len(out) != 28:
        raise RuntimeError("Healthcare policy must contain 28 factors.")
    return out


def generate_controlled_documentary_evidence(client_ids: List[str], evidence_seed: int):
    if len(client_ids) != 10:
        raise RuntimeError("Experiment E frozen documentary design requires K=10.")
    bundle_ids = [f"E{i:02d}" for i in range(1, 11)]
    rng = np.random.default_rng(int(evidence_seed))
    assignment = rng.permutation(len(bundle_ids))
    rows = []
    evidence = {}
    for pos, cid in enumerate(client_ids):
        bid = bundle_ids[int(assignment[pos])]
        bundle = BUNDLE_TEMPLATES[bid]
        evidence[cid] = {dim: dict(vals) for dim, vals in bundle["factors"].items()}
        for dim in DOCUMENTARY_DIMS:
            for factor, score in evidence[cid][dim].items():
                key = (dim, factor)
                rows.append({
                    "client": str(cid),
                    "bundle_id": bid,
                    "evidence_profile": bundle["profile"],
                    "scenario_role": bundle["profile"],
                    "profile_variant": int(bundle["variant"]),
                    "dimension": dim,
                    "dimension_name": DIMENSION_NAMES[dim],
                    "factor": factor,
                    "evidence_source_type": "CONTROLLED_DOCUMENTARY_EVIDENCE",
                    "rubric_score_0_5": float(score),
                    "adequacy_min_rank": float(FACTOR_ADEQUACY_MIN[dim][factor]),
                    "is_mandatory_factor": bool(key in MANDATORY_MINIMA),
                    "mandatory_min_rank": float(MANDATORY_MINIMA[key]) if key in MANDATORY_MINIMA else np.nan,
                    "meets_mandatory_requirement": (
                        bool(float(score) >= MANDATORY_MINIMA[key]) if key in MANDATORY_MINIMA else np.nan
                    ),
                    "is_review_factor": bool(key in REVIEW_FACTORS),
                    "review_acceptance_threshold_0_5": (
                        REFERENCE_REVIEW_CUT if key in REVIEW_FACTORS else np.nan
                    ),
                    "evidence_seed": int(evidence_seed),
                    "domain": DOMAIN,
                })
    return evidence, pd.DataFrame(rows)


def evidence_assignment_summary(evidence_df: pd.DataFrame) -> pd.DataFrame:
    return (
        evidence_df.groupby(["client", "bundle_id"], as_index=False)
        .agg(
            evidence_profile=("evidence_profile", "first"),
            scenario_role=("scenario_role", "first"),
            profile_variant=("profile_variant", "first"),
            documentary_factor_count=("factor", "count"),
            documentary_mean_score=("rubric_score_0_5", "mean"),
            documentary_min_score=("rubric_score_0_5", "min"),
            documentary_max_score=("rubric_score_0_5", "max"),
            evidence_seed=("evidence_seed", "first"),
        )
        .sort_values("client").reset_index(drop=True)
    )


def build_full_factor_table(
    client_ids: List[str],
    documentary_evidence: Dict[str, Dict[str, Dict[str, float]]],
    dq_scores: Dict[str, Dict[str, float]],
    policy: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    policy_index = policy.set_index(["dimension", "factor"])
    rows = []
    for cid in client_ids:
        for dim in DIMS:
            source_scores = dq_scores[cid] if dim == "dim2" else documentary_evidence[cid][dim]
            for factor in FACTOR_NAMES[dim]:
                score = float(source_scores[factor])
                p = policy_index.loc[(dim, factor)]
                rows.append({
                    "client": str(cid),
                    "domain": DOMAIN,
                    "dimension": dim,
                    "dimension_name": DIMENSION_NAMES[dim],
                    "factor": factor,
                    "factor_source": (
                        "MACHINE_MEASURED_TRAIN_ONLY" if dim == "dim2"
                        else "CONTROLLED_DOCUMENTARY_EVIDENCE"
                    ),
                    "rubric_score_0_5": score,
                    "adequacy_min_rank": float(p["adequacy_min_rank"]),
                    "is_mandatory_factor": bool(p["is_mandatory_factor"]),
                    "mandatory_minimum_0_5": (
                        float(p["mandatory_minimum_0_5"])
                        if pd.notna(p["mandatory_minimum_0_5"]) else np.nan
                    ),
                    "is_review_factor": bool(p["is_review_factor"]),
                    "review_acceptance_threshold_0_5": (
                        float(p["review_acceptance_threshold_0_5"])
                        if pd.notna(p["review_acceptance_threshold_0_5"]) else np.nan
                    ),
                    "evidence_seed": int(FROZEN_EVIDENCE_ASSIGNMENT_SEED),
                })
    f = pd.DataFrame(rows)
    if len(f) != len(client_ids) * 28:
        raise RuntimeError("Full factor table must contain 28 rows per client.")
    dim_scores = (
        f.groupby(["client", "dimension"])["rubric_score_0_5"]
        .mean().unstack("dimension").reindex(columns=DIMS).sort_index()
    )
    return f, dim_scores


def evaluate_policy_v20_2(
    factor_table: pd.DataFrame,
    dim_scores: pd.DataFrame,
    hps_weights: Dict[str, float],
    lower_cut: float,
    upper_cut: float,
    review_cut: float,
    dimension_floor: float,
    removed_dim: Optional[str] = None,
) -> Tuple[pd.DataFrame, dict]:
    lower_cut = float(lower_cut)
    upper_cut = float(upper_cut)
    review_cut = float(review_cut)
    dimension_floor = float(dimension_floor)
    if removed_dim is not None and removed_dim not in DIMS:
        raise ValueError(f"Unknown removed dimension: {removed_dim}")
    if not (0.0 <= lower_cut < upper_cut <= 5.0):
        raise ValueError("Invalid HPS thresholds: require 0 <= lower < upper <= 5.")
    if not (0.0 <= review_cut <= 5.0):
        raise ValueError("Review threshold must be in [0,5].")
    if not (0.0 <= dimension_floor <= 5.0):
        raise ValueError("Dimension floor must be in [0,5].")

    active_dims = [d for d in DIMS if d != removed_dim]
    raw_weights = {d: float(hps_weights[d]) for d in active_dims}
    wsum = float(sum(raw_weights.values()))
    norm_weights = {d: w / wsum for d, w in raw_weights.items()}

    active_factors = factor_table.loc[factor_table["dimension"].isin(active_dims)].copy()
    mandatory = active_factors.loc[bool_series(active_factors["is_mandatory_factor"])].copy()
    review = active_factors.loc[bool_series(active_factors["is_review_factor"])].copy()

    rows = []
    for client in dim_scores.index.astype(str):
        ds = dim_scores.loc[client]
        hps = float(sum(float(ds[d]) * norm_weights[d] for d in active_dims))

        m = mandatory.loc[mandatory["client"].astype(str).eq(client)].copy()
        mandatory_missing, mandatory_below = [], []
        for r in m.itertuples(index=False):
            score = float(r.rubric_score_0_5)
            minimum = float(r.mandatory_minimum_0_5)
            key = f"{r.dimension}.{r.factor}"
            if not np.isfinite(score):
                mandatory_missing.append(key)
            elif score < minimum:
                mandatory_below.append(f"{key}:{score:.1f}<{minimum:.1f}")
        mandatory_pass = (not mandatory_missing and not mandatory_below)

        floor_failures = [
            d for d in active_dims
            if (not np.isfinite(float(ds[d]))) or float(ds[d]) < dimension_floor
        ]

        rv = review.loc[review["client"].astype(str).eq(client), "rubric_score_0_5"]
        review_values = pd.to_numeric(rv, errors="coerce").to_numpy(float)
        review_missing = int(np.sum(~np.isfinite(review_values)))
        review_score = (
            float(np.mean(review_values[np.isfinite(review_values)]))
            if np.any(np.isfinite(review_values)) else np.nan
        )

        if not mandatory_pass:
            final_action, decision_path, status = (
                "REJECT", "MANDATORY_GOVERNANCE_GATE", "AUTO_REJECTED_MANDATORY_GATE"
            )
        elif floor_failures:
            final_action, decision_path, status = (
                "REJECT", "DIMENSION_FLOOR", "AUTO_REJECTED_DIMENSION_FLOOR"
            )
        elif hps < lower_cut:
            final_action, decision_path, status = (
                "REJECT", "LOW_HPS", "AUTO_REJECTED_LOW_HPS"
            )
        elif hps >= upper_cut:
            final_action, decision_path, status = (
                "ACCEPT", "DIRECT_AUTO_ACCEPT", "DIRECT_AUTO_ACCEPTED"
            )
        else:
            decision_path = "HPS_REVIEW_BAND"
            if review_missing > 0 or not np.isfinite(review_score):
                final_action, status = "REJECT", "AUTO_REJECTED_REVIEW_MISSING_EVIDENCE"
            elif review_score >= review_cut:
                final_action, status = "ACCEPT", "ACCEPTED_AFTER_AUTOMATED_REVIEW"
            else:
                final_action, status = "REJECT", "AUTO_REJECTED_REVIEW_SCORE"

        rows.append({
            "client": client,
            "hps": hps,
            "mandatory_gate_pass": bool(mandatory_pass),
            "mandatory_factor_count": int(len(m)),
            "mandatory_missing": ";".join(mandatory_missing),
            "mandatory_below_minimum": ";".join(mandatory_below),
            "review_score_0_5": review_score,
            "review_factor_count": int(len(review_values)),
            "review_missing_count": int(review_missing),
            "dimension_floor_failures": ";".join(floor_failures),
            "decision_path": decision_path,
            "status": status,
            "final_action": final_action,
            "lower_cut": lower_cut,
            "upper_cut": upper_cut,
            "review_cut": review_cut,
            "dimension_floor": dimension_floor,
            "removed_dimension": removed_dim or "",
            **{f"{d}_score_0_5": float(ds[d]) for d in DIMS},
        })

    return pd.DataFrame(rows), {
        "active_dimensions": active_dims,
        "renormalized_hps_weights": norm_weights,
        "active_mandatory_factor_count": int(mandatory[["dimension", "factor"]].drop_duplicates().shape[0]),
        "active_review_factor_count": int(review[["dimension", "factor"]].drop_duplicates().shape[0]),
        "removed_dimension": removed_dim,
    }


def assert_fresh_reference_design(reference: pd.DataFrame):
    x = reference.sort_values("client").reset_index(drop=True)
    actions = dict(zip(x["client"].astype(str), x["final_action"].astype(str)))
    paths = dict(zip(x["client"].astype(str), x["decision_path"].astype(str)))
    if actions != EXPECTED_REFERENCE_ACTIONS:
        raise RuntimeError(
            "Fresh reference governance does not reproduce the frozen final design.\n"
            + x[["client", "hps", "review_score_0_5", "decision_path", "final_action"]].to_string(index=False)
        )
    if paths != EXPECTED_REFERENCE_PATHS:
        raise RuntimeError(
            "Fresh reference decision paths differ from the frozen final design.\n"
            + x[["client", "decision_path", "status"]].to_string(index=False)
        )
    admitted = x.loc[x["final_action"].eq("ACCEPT"), "client"].astype(str).tolist()
    if admitted != EXPECTED_REFERENCE_COHORT:
        raise RuntimeError(f"Expected reference cohort {EXPECTED_REFERENCE_COHORT}, got {admitted}")
    print("Fresh reference governance self-check: PASS")
    print(f"Fresh reference admitted cohort: {admitted}")
    return x

# ======================================================================================
# 6. POLICY SENSITIVITY + LODO CONFIGURATIONS
# ======================================================================================

def build_sensitivity_configurations() -> pd.DataFrame:
    rows = [{
        "configuration": "REFERENCE",
        "family": "REFERENCE",
        "perturbation_pct": 0,
        "lower_cut": REFERENCE_LOWER_CUT,
        "upper_cut": REFERENCE_UPPER_CUT,
        "review_cut": REFERENCE_REVIEW_CUT,
        "dimension_floor": REFERENCE_DIMENSION_FLOOR,
        "removed_dimension": "",
        "valid_policy": True,
        "invalid_reason": "",
    }]

    def add_threshold_config(name, family, pct, lower, upper, review, floor):
        valid = bool(
            0.0 <= lower <= 5.0 and 0.0 <= upper <= 5.0 and lower < upper
            and 0.0 <= review <= 5.0 and 0.0 <= floor <= 5.0
        )
        reason = "" if valid else (
            "Invalid policy range/order: require 0<=lower<upper<=5, "
            "0<=review<=5, and 0<=floor<=5."
        )
        rows.append({
            "configuration": name,
            "family": family,
            "perturbation_pct": int(pct),
            "lower_cut": float(lower),
            "upper_cut": float(upper),
            "review_cut": float(review),
            "dimension_floor": float(floor),
            "removed_dimension": "",
            "valid_policy": valid,
            "invalid_reason": reason,
        })

    for pct in PERTURBATION_PCTS:
        f = 1.0 + pct / 100.0

        add_threshold_config(
            f"HPS_LOWER_{pct:+d}pct", "HPS_LOWER", pct,
            REFERENCE_LOWER_CUT * f, REFERENCE_UPPER_CUT,
            REFERENCE_REVIEW_CUT, REFERENCE_DIMENSION_FLOOR,
        )
        add_threshold_config(
            f"HPS_UPPER_{pct:+d}pct", "HPS_UPPER", pct,
            REFERENCE_LOWER_CUT, REFERENCE_UPPER_CUT * f,
            REFERENCE_REVIEW_CUT, REFERENCE_DIMENSION_FLOOR,
        )
        add_threshold_config(
            f"HPS_BOTH_{pct:+d}pct", "HPS_BOTH", pct,
            REFERENCE_LOWER_CUT * f, REFERENCE_UPPER_CUT * f,
            REFERENCE_REVIEW_CUT, REFERENCE_DIMENSION_FLOOR,
        )
        add_threshold_config(
            f"REVIEW_THRESHOLD_{pct:+d}pct", "REVIEW_THRESHOLD", pct,
            REFERENCE_LOWER_CUT, REFERENCE_UPPER_CUT,
            REFERENCE_REVIEW_CUT * f, REFERENCE_DIMENSION_FLOOR,
        )
        add_threshold_config(
            f"DIMENSION_FLOOR_{pct:+d}pct", "DIMENSION_FLOOR", pct,
            REFERENCE_LOWER_CUT, REFERENCE_UPPER_CUT,
            REFERENCE_REVIEW_CUT, REFERENCE_DIMENSION_FLOOR * f,
        )

    for dim in DIMS:
        rows.append({
            "configuration": f"LODO_{dim}",
            "family": "LODO",
            "perturbation_pct": 0,
            "lower_cut": REFERENCE_LOWER_CUT,
            "upper_cut": REFERENCE_UPPER_CUT,
            "review_cut": REFERENCE_REVIEW_CUT,
            "dimension_floor": REFERENCE_DIMENSION_FLOOR,
            "removed_dimension": dim,
            "valid_policy": True,
            "invalid_reason": "",
        })

    return pd.DataFrame(rows)


def evaluate_all_configurations(
    configs: pd.DataFrame,
    factor_table: pd.DataFrame,
    dim_scores: pd.DataFrame,
    hps_weights: Dict[str, float],
):
    reference, _ = evaluate_policy_v20_2(
        factor_table, dim_scores, hps_weights,
        REFERENCE_LOWER_CUT, REFERENCE_UPPER_CUT,
        REFERENCE_REVIEW_CUT, REFERENCE_DIMENSION_FLOOR,
        removed_dim=None,
    )
    reference = reference.sort_values("client").reset_index(drop=True)
    ref_accept = reference.loc[
        reference["final_action"].eq("ACCEPT"), "client"
    ].astype(str).tolist()
    ref_set = set(ref_accept)

    summary_rows = []
    client_frames = []

    for cfg in configs.itertuples(index=False):
        base = {
            "configuration": str(cfg.configuration),
            "family": str(cfg.family),
            "perturbation_pct": int(cfg.perturbation_pct),
            "lower_cut": float(cfg.lower_cut),
            "upper_cut": float(cfg.upper_cut),
            "review_cut": float(cfg.review_cut),
            "dimension_floor": float(cfg.dimension_floor),
            "removed_dimension": str(cfg.removed_dimension),
            "valid_policy": bool(cfg.valid_policy),
            "invalid_reason": str(cfg.invalid_reason),
        }

        if not bool(cfg.valid_policy):
            summary_rows.append({
                **base,
                "accepted_count": np.nan,
                "accepted_rate": np.nan,
                "admitted_clients": "",
                "gained_count": np.nan,
                "gained_clients": "",
                "lost_count": np.nan,
                "lost_clients": "",
                "net_admission_change": np.nan,
                "admission_rate_delta_pp": np.nan,
                "decision_flip_count": np.nan,
                "decision_flip_rate": np.nan,
                "accepted_set_jaccard_vs_reference": np.nan,
                "cohort_key": "",
                "renormalized_hps_weights": "",
                "active_mandatory_factor_count": np.nan,
                "active_review_factor_count": np.nan,
            })
            continue

        removed = str(cfg.removed_dimension).strip() or None
        current, meta = evaluate_policy_v20_2(
            factor_table=factor_table,
            dim_scores=dim_scores,
            hps_weights=hps_weights,
            lower_cut=float(cfg.lower_cut),
            upper_cut=float(cfg.upper_cut),
            review_cut=float(cfg.review_cut),
            dimension_floor=float(cfg.dimension_floor),
            removed_dim=removed,
        )
        current = current.sort_values("client").reset_index(drop=True)
        current.insert(0, "configuration", str(cfg.configuration))
        current.insert(1, "family", str(cfg.family))
        client_frames.append(current)

        admitted = current.loc[
            current["final_action"].eq("ACCEPT"), "client"
        ].astype(str).tolist()
        cur_set = set(admitted)
        gained = sorted(cur_set - ref_set)
        lost = sorted(ref_set - cur_set)

        comp = reference[["client", "final_action"]].merge(
            current[["client", "final_action"]], on="client",
            suffixes=("_reference", "_current"), validate="one_to_one"
        )
        flips = comp["final_action_reference"].ne(comp["final_action_current"])

        summary_rows.append({
            **base,
            "accepted_count": int(len(admitted)),
            "accepted_rate": float(len(admitted) / K_SUBMISSIONS),
            "admitted_clients": ";".join(admitted),
            "gained_count": int(len(gained)),
            "gained_clients": ";".join(gained),
            "lost_count": int(len(lost)),
            "lost_clients": ";".join(lost),
            "net_admission_change": int(len(gained) - len(lost)),
            "admission_rate_delta_pp": float(
                100.0 * (len(admitted) - len(ref_accept)) / K_SUBMISSIONS
            ),
            "decision_flip_count": int(flips.sum()),
            "decision_flip_rate": float(flips.mean()),
            "accepted_set_jaccard_vs_reference": jaccard(ref_accept, admitted),
            "cohort_key": ";".join(sorted(admitted)),
            "renormalized_hps_weights": json.dumps(
                meta["renormalized_hps_weights"], sort_keys=True
            ),
            "active_mandatory_factor_count": int(meta["active_mandatory_factor_count"]),
            "active_review_factor_count": int(meta["active_review_factor_count"]),
        })

    client_level = pd.concat(client_frames, ignore_index=True) if client_frames else pd.DataFrame()
    return pd.DataFrame(summary_rows), client_level, reference


def assign_cohort_ids(governance_summary: pd.DataFrame):
    out = governance_summary.copy()
    ref_key = ";".join(sorted(EXPECTED_REFERENCE_COHORT))

    keys = sorted(
        out.loc[
            out["valid_policy"].eq(True) & out["cohort_key"].astype(str).ne(""),
            "cohort_key"
        ].unique().tolist()
    )

    mapping = {ref_key: "REFERENCE_COHORT"}
    counter = 1
    for key in keys:
        if key == ref_key:
            continue
        mapping[key] = f"COHORT_{counter:02d}"
        counter += 1

    out["cohort_id"] = out["cohort_key"].map(mapping).fillna("")
    return out, mapping


# ======================================================================================
# 7. DIABETES DATA PREPARATION + TRAIN-ONLY DATA QUALITY — SAME CORE DESIGN AS A1
# ======================================================================================

TARGET = "readmitted"
ID_COLUMNS = ["encounter_id", "patient_nbr"]


def load_diabetes(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df = df.replace("?", np.nan)
    if TARGET not in df.columns:
        raise RuntimeError(f"Missing target column: {TARGET}")
    mapping = {"NO": 0, ">30": 1, "<30": 2}
    df = df[df[TARGET].isin(mapping)].copy()
    df["_target"] = df[TARGET].map(mapping).astype(np.int32)
    df["_row_id"] = np.arange(len(df), dtype=np.int64)
    return df


def global_patient_grouped_split(df: pd.DataFrame, seed: int):
    if "patient_nbr" in df.columns:
        splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=int(seed))
        train_idx, test_idx = next(
            splitter.split(
                np.zeros(len(df)), y=df["_target"].to_numpy(),
                groups=df["patient_nbr"].astype(str).to_numpy(),
            )
        )
        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()
        overlap = len(set(train_df["patient_nbr"].astype(str)) & set(test_df["patient_nbr"].astype(str)))
        if overlap != 0:
            raise RuntimeError("Patient leakage detected across TRAIN/TEST.")
    else:
        train_df, test_df = train_test_split(
            df, test_size=0.20, stratify=df["_target"], random_state=int(seed)
        )
        overlap = 0
    return train_df.reset_index(drop=True), test_df.reset_index(drop=True), overlap


def dirichlet_partition_dataframe(
    train_df: pd.DataFrame, n_clients: int, alpha: float, seed: int,
    min_client_records: int = 100,
) -> Dict[str, pd.DataFrame]:
    y = train_df["_target"].to_numpy()
    client_ids = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")[:n_clients]
    for attempt in range(100):
        rng = np.random.default_rng(int(seed + attempt))
        buckets = [[] for _ in range(n_clients)]
        for cls in sorted(np.unique(y)):
            idx = np.where(y == cls)[0]
            rng.shuffle(idx)
            props = rng.dirichlet(np.full(n_clients, float(alpha)))
            cuts = (np.cumsum(props)[:-1] * len(idx)).astype(int)
            splits = np.split(idx, cuts)
            for k, s in enumerate(splits):
                buckets[k].extend(s.tolist())
        if min(len(b) for b in buckets) >= int(min_client_records):
            return {
                cid: train_df.iloc[np.asarray(idxs, dtype=int)].copy()
                for cid, idxs in zip(client_ids, buckets)
            }
    raise RuntimeError("Could not obtain a valid Dirichlet client partition.")


class TabularPreprocessor:
    def __init__(self, feature_cols, numeric_cols, categorical_cols, mean, std, categories, encoder):
        self.feature_cols = feature_cols
        self.numeric_cols = numeric_cols
        self.categorical_cols = categorical_cols
        self.mean = mean
        self.std = std
        self.categories = categories
        self.encoder = encoder

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        pieces = []
        if self.numeric_cols:
            x_num = []
            for c in self.numeric_cols:
                s = pd.to_numeric(df[c], errors="coerce").astype(float)
                a = s.fillna(self.mean[c]).to_numpy(dtype=np.float32)
                a = (a - self.mean[c]) / self.std[c]
                x_num.append(a[:, None])
            pieces.append(np.concatenate(x_num, axis=1).astype(np.float32))
        if self.categorical_cols:
            cat = pd.DataFrame({
                c: df[c].astype("string").fillna("__MISSING__").astype(str)
                for c in self.categorical_cols
            })
            pieces.append(np.asarray(self.encoder.transform(cat), dtype=np.float32))
        if not pieces:
            raise RuntimeError("No predictor columns remained.")
        return np.concatenate(pieces, axis=1).astype(np.float32)


def fit_train_only_preprocessor(client_frames: Dict[str, pd.DataFrame]) -> TabularPreprocessor:
    any_df = next(iter(client_frames.values()))
    feature_cols = [
        c for c in any_df.columns
        if c not in {TARGET, "_target", "_row_id", *ID_COLUMNS}
    ]
    numeric_cols, categorical_cols = [], []
    for c in feature_cols:
        total_nonmissing, numeric_valid = 0, 0
        for df in client_frames.values():
            raw = df[c]
            nm = raw.notna()
            total_nonmissing += int(nm.sum())
            if nm.any():
                numeric_valid += int(pd.to_numeric(raw[nm], errors="coerce").notna().sum())
        ratio = numeric_valid / max(1, total_nonmissing)
        (numeric_cols if ratio >= 0.95 else categorical_cols).append(c)

    mean, std = {}, {}
    for c in numeric_cols:
        count, sum_, sumsq = 0, 0.0, 0.0
        for df in client_frames.values():
            a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
            a = a[np.isfinite(a)]
            count += len(a)
            sum_ += float(a.sum())
            sumsq += float(np.square(a).sum())
        mu = sum_ / max(1, count)
        var = max(1e-12, sumsq / max(1, count) - mu * mu)
        mean[c], std[c] = float(mu), float(math.sqrt(var))

    categories = {}
    for c in categorical_cols:
        vals = set()
        for df in client_frames.values():
            vals.update(df[c].astype("string").fillna("__MISSING__").astype(str).unique().tolist())
        categories[c] = sorted(vals)

    encoder = OneHotEncoder(
        categories=[categories[c] for c in categorical_cols],
        handle_unknown="ignore", sparse_output=False, dtype=np.float32,
    )
    if categorical_cols:
        max_len = max(len(categories[c]) for c in categorical_cols)
        dummy = {c: [categories[c][i % len(categories[c])] for i in range(max_len)] for c in categorical_cols}
        encoder.fit(pd.DataFrame(dummy))

    return TabularPreprocessor(feature_cols, numeric_cols, categorical_cols, mean, std, categories, encoder)


def score_lower_is_better(value: float, cuts: List[float]) -> float:
    for score, upper in zip([5, 4, 3, 2, 1], cuts):
        if float(value) <= float(upper):
            return float(score)
    return 0.0


def score_higher_is_better(value: float, cuts: List[float]) -> float:
    for score, lower in zip([5, 4, 3, 2, 1], cuts):
        if float(value) >= float(lower):
            return float(score)
    return 0.0


def js_divergence(p, q, eps=1e-12) -> float:
    p, q = np.asarray(p, dtype=float), np.asarray(q, dtype=float)
    p = p / max(eps, p.sum())
    q = q / max(eps, q.sum())
    m = 0.5 * (p + q)
    kl_pm = np.sum(np.where(p > 0, p * np.log((p + eps) / (m + eps)), 0.0))
    kl_qm = np.sum(np.where(q > 0, q * np.log((q + eps) / (m + eps)), 0.0))
    return float(0.5 * (kl_pm + kl_qm))


def build_tabular_reference(client_frames, preprocessor):
    ref = {"num_hist": {}, "cat_values": {}}
    for c in preprocessor.numeric_cols[:12]:
        local_min, local_max = [], []
        for df in client_frames.values():
            a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
            a = a[np.isfinite(a)]
            if len(a):
                local_min.append(float(np.min(a)))
                local_max.append(float(np.max(a)))
        if not local_min:
            continue
        gmin, gmax = float(min(local_min)), float(max(local_max))
        edges = (
            np.array([gmin - 1e-6, gmax + 1e-6], dtype=float)
            if gmax <= gmin else np.linspace(gmin, gmax, 11, dtype=float)
        )
        global_hist = np.zeros(len(edges) - 1, dtype=np.float64)
        for df in client_frames.values():
            a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
            a = a[np.isfinite(a)]
            if len(a):
                h, _ = np.histogram(a, bins=edges)
                global_hist += h.astype(np.float64)
        ref["num_hist"][c] = {"edges": edges.tolist(), "hist": global_hist.tolist()}
    for c in preprocessor.categorical_cols:
        values = set()
        for df in client_frames.values():
            values.update(df[c].astype("string").fillna("__MISSING__").astype(str).unique().tolist())
        ref["cat_values"][c] = sorted(values)
    return ref


def tabular_dq_scores(df: pd.DataFrame, preprocessor, reference):
    feature_df = df[preprocessor.feature_cols]
    missing_fraction = float(feature_df.isna().mean().mean())
    duplicate_fraction = float(feature_df.duplicated().mean())

    bad, observed = 0, 0
    for c in preprocessor.numeric_cols:
        raw = df[c]
        nm = raw.notna()
        observed += int(nm.sum())
        if nm.any():
            conv = pd.to_numeric(raw[nm], errors="coerce").to_numpy(dtype=float)
            bad += int(np.sum(~np.isfinite(conv)))
    for c in preprocessor.categorical_cols:
        raw = df[c]
        nm = raw.notna()
        observed += int(nm.sum())
        if nm.any():
            bad += int(np.sum(raw[nm].astype(str).str.strip().eq("")))
    error_fraction = float(bad / max(1, observed))

    type_bad, type_obs = 0, 0
    for c in preprocessor.numeric_cols:
        raw = df[c]
        nm = raw.notna()
        type_obs += int(nm.sum())
        if nm.any():
            type_bad += int(pd.to_numeric(raw[nm], errors="coerce").isna().sum())
    type_inconsistency = float(type_bad / max(1, type_obs))

    invalid_label_fraction = float((~df["_target"].isin([0, 1, 2])).mean())

    jsds = []
    for c, spec in reference["num_hist"].items():
        a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
        a = a[np.isfinite(a)]
        if len(a):
            hist, _ = np.histogram(a, bins=np.asarray(spec["edges"], dtype=float))
            jsds.append(js_divergence(hist, np.asarray(spec["hist"], dtype=float)))
    max_jsd = float(max(jsds)) if jsds else 0.0

    coverage_vals = []
    for c, ref_vals in reference["cat_values"].items():
        ref_set = set(ref_vals)
        if ref_set:
            client_set = set(df[c].astype("string").fillna("__MISSING__").astype(str).unique())
            coverage_vals.append(len(client_set & ref_set) / len(ref_set))
    mean_coverage = float(np.mean(coverage_vals)) if coverage_vals else 1.0

    n = len(df)
    record_violation = np.zeros(n, dtype=bool)
    if "encounter_id" not in df.columns:
        record_violation[:] = True
    else:
        encounter = df["encounter_id"]
        record_violation |= encounter.isna().to_numpy()
        record_violation |= encounter.duplicated(keep=False).to_numpy()
    for c in [
        "time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
        "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses",
    ]:
        if c in df.columns:
            a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
            record_violation |= (~np.isfinite(a)) | (a < 0)
    structural_violation_fraction = float(np.mean(record_violation)) if n else 1.0

    scores = {
        "completeness": score_lower_is_better(missing_fraction, [0.01, 0.05, 0.10, 0.20, 0.50]),
        "duplication_rate": score_lower_is_better(duplicate_fraction, [0.01, 0.02, 0.05, 0.10, 0.20]),
        "value_validity_error_rate": score_lower_is_better(error_fraction, [0.01, 0.02, 0.05, 0.10, 0.15]),
        "type_consistency": score_lower_is_better(type_inconsistency, [0.01, 0.02, 0.05, 0.10, 0.20]),
        "label_integrity": score_lower_is_better(invalid_label_fraction, [0.001, 0.01, 0.02, 0.05, 0.10]),
        "feature_distribution_consistency": score_lower_is_better(max_jsd, [0.01, 0.025, 0.05, 0.10, 0.20]),
        "feature_category_coverage": score_higher_is_better(mean_coverage, [0.90, 0.825, 0.75, 0.65, 0.50]),
        "structural_constraint_integrity": score_lower_is_better(structural_violation_fraction, [0.001, 0.01, 0.02, 0.05, 0.10]),
    }
    raw = {
        "missing_fraction": missing_fraction,
        "duplicate_fraction": duplicate_fraction,
        "error_fraction": error_fraction,
        "type_inconsistency_fraction": type_inconsistency,
        "invalid_label_fraction": invalid_label_fraction,
        "max_jsd": max_jsd,
        "mean_category_coverage": mean_coverage,
        "structural_violation_fraction": structural_violation_fraction,
    }
    return scores, raw


def class_weight_dict(y: np.ndarray) -> Dict[int, float]:
    y = np.asarray(y, dtype=np.int32)
    classes = np.unique(y)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return {int(c): float(w) for c, w in zip(classes, weights)}


def client_partition_table(clients_raw: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    for cid, df in clients_raw.items():
        counts = df["_target"].value_counts().to_dict()
        rows.append({
            "client": str(cid), "records": int(len(df)),
            "class_0_NO": int(counts.get(0, 0)),
            "class_1_GT30": int(counts.get(1, 0)),
            "class_2_LT30": int(counts.get(2, 0)),
        })
    return pd.DataFrame(rows).sort_values("client").reset_index(drop=True)


def prepare_diabetes_for_experiment_e(csv_path: str) -> dict:
    raw = load_diabetes(csv_path)
    train_df, test_df, patient_overlap = global_patient_grouped_split(raw, GLOBAL_SPLIT_SEED)
    clients = dirichlet_partition_dataframe(
        train_df, K_SUBMISSIONS, DIRICHLET_ALPHA, CLIENT_PARTITION_SEED
    )
    current_audit = client_partition_table(clients)
    if current_audit["client"].astype(str).tolist() != EXPECTED_CLIENTS:
        raise RuntimeError("Unexpected K=10 client identities.")

    pre = fit_train_only_preprocessor(clients)
    reference = build_tabular_reference(clients, pre)
    client_arrays, dq_scores, dq_rows = {}, {}, []
    for cid, df in clients.items():
        scores, raw_metrics = tabular_dq_scores(df, pre, reference)
        dq_scores[cid] = scores
        dq_rows.append({"client": cid, **scores, **raw_metrics})
        client_arrays[cid] = (
            pre.transform(df), df["_target"].to_numpy(dtype=np.int32)
        )

    X_test = pre.transform(test_df)
    y_test = test_df["_target"].to_numpy(dtype=np.int32)
    y_train_all = np.concatenate([client_arrays[c][1] for c in sorted(client_arrays)])
    cw = class_weight_dict(y_train_all)

    train_ids, test_ids = set(train_df["_row_id"].astype(str)), set(test_df["_row_id"].astype(str))
    client_union, duplicate_across_clients = set(), 0
    for df in clients.values():
        ids = set(df["_row_id"].astype(str))
        duplicate_across_clients += len(client_union & ids)
        client_union |= ids
    leakage = {
        "train_test_overlap": len(train_ids & test_ids),
        "test_rows_in_any_client": len(test_ids & client_union),
        "train_rows_missing_from_clients": len(train_ids - client_union),
        "client_rows_not_in_global_train": len(client_union - train_ids),
        "duplicate_train_rows_across_clients": duplicate_across_clients,
        "patient_overlap": int(patient_overlap),
    }
    leakage["pass"] = all(int(leakage[k]) == 0 for k in leakage if k != "pass")
    if not leakage["pass"]:
        raise RuntimeError(f"FAIL-CLOSED leakage audit failed: {leakage}")

    return {
        "raw": raw, "train_df": train_df, "test_df": test_df,
        "clients_raw": clients, "client_arrays": client_arrays,
        "client_ids": sorted(client_arrays), "X_test": X_test, "y_test": y_test,
        "class_weights": cw, "preprocessor": pre, "input_dim": int(X_test.shape[1]),
        "partition_audit": current_audit, "leakage_audit": pd.DataFrame([leakage]),
        "dq_scores": dq_scores, "dq_audit": pd.DataFrame(dq_rows),
        "dataset_metadata": pd.DataFrame([{
            "raw_rows": int(len(raw)), "train_rows": int(len(train_df)), "test_rows": int(len(test_df)),
            "patient_overlap": int(patient_overlap), "input_dim": int(X_test.shape[1]),
            "n_clients": K_SUBMISSIONS, "numeric_features": len(pre.numeric_cols),
            "categorical_features": len(pre.categorical_cols),
        }]),
    }

# ======================================================================================
# 8. MODEL + TADP-VR FEDERATED TRAINING
# ======================================================================================

def build_diabetes_model(input_dim: int, lr: float = LEARNING_RATE) -> keras.Model:
    inp = keras.Input(shape=(int(input_dim),), dtype=tf.float32)
    x = layers.Dense(128, activation="relu")(inp)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.20)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.20)(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(3, activation="softmax", dtype=tf.float32)(x)
    model = keras.Model(inp, out)
    model.optimizer = keras.optimizers.Adam(learning_rate=float(lr))
    return model


def model_parameter_bytes(model: keras.Model) -> int:
    return int(sum(np.asarray(w).nbytes for w in model.get_weights()))


def natural_steps(n_records: int, batch_size: int, local_epochs: int = 1) -> int:
    return int(max(1, math.ceil(int(n_records) / int(batch_size)) * int(local_epochs)))


def train_exact_steps(
    model: keras.Model,
    X: np.ndarray,
    y: np.ndarray,
    steps: int,
    batch_size: int,
    seed: int,
    class_weights: Optional[Dict[int, float]] = None,
):
    steps = int(max(1, steps))
    rng = np.random.default_rng(int(seed))
    n = len(y)
    loss_fn = keras.losses.SparseCategoricalCrossentropy(
        reduction=keras.losses.Reduction.NONE
    )

    order = rng.permutation(n)
    cursor = 0
    for _ in range(steps):
        if cursor + batch_size > n:
            order = rng.permutation(n)
            cursor = 0
        idx = order[cursor:cursor + batch_size]
        cursor += batch_size

        xb = tf.convert_to_tensor(np.asarray(X[idx]), dtype=tf.float32)
        yb_np = np.asarray(y[idx], dtype=np.int32)
        yb = tf.convert_to_tensor(yb_np, dtype=tf.int32)

        with tf.GradientTape() as tape:
            probs = model(xb, training=True)
            per_loss = loss_fn(yb, probs)
            if class_weights:
                sw = np.asarray([class_weights.get(int(v), 1.0) for v in yb_np], dtype=np.float32)
                sw_t = tf.convert_to_tensor(sw)
                loss = tf.reduce_sum(per_loss * sw_t) / tf.reduce_sum(sw_t)
            else:
                loss = tf.reduce_mean(per_loss)

        grads = tape.gradient(loss, model.trainable_variables)
        model.optimizer.apply_gradients(zip(grads, model.trainable_variables))


def aggregate_weights(local_weights, sample_sizes):
    sizes = np.asarray(sample_sizes, dtype=float)
    alpha = sizes / sizes.sum()
    out = []
    for layer_idx in range(len(local_weights[0])):
        x = sum(alpha[j] * np.asarray(local_weights[j][layer_idx])
                for j in range(len(local_weights)))
        out.append(np.asarray(x))
    return out


def evaluate_model(model: keras.Model, X: np.ndarray, y: np.ndarray) -> Dict[str, float]:
    p = model.predict(X, batch_size=512, verbose=0)
    pred = np.argmax(p, axis=1)
    out = {
        "accuracy": float(accuracy_score(y, pred)),
        "precision_macro": float(precision_score(y, pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y, pred, average="macro", zero_division=0)),
        "f1_macro": float(f1_score(y, pred, average="macro", zero_division=0)),
    }
    try:
        out["roc_auc_ovr_macro"] = float(
            roc_auc_score(y, p, multi_class="ovr", average="macro")
        )
    except Exception:
        out["roc_auc_ovr_macro"] = np.nan
    return out


def train_tadp_vr_cohort(
    selected_clients: List[str],
    data: dict,
    training_seed: int,
) -> dict:
    selected_clients = sorted(map(str, selected_clients))
    if not selected_clients:
        raise RuntimeError("Cannot train an empty admitted cohort.")

    # Same W0 for every cohort within a seed. Resetting the seed per cohort makes
    # sensitivity comparisons order-independent and paired on the same stochastic seed.
    seed_everything(training_seed)
    base = build_diabetes_model(data["input_dim"], LEARNING_RATE)
    initial_weights = [np.array(w, copy=True) for w in base.get_weights()]
    initial_hash = sha256_weights(initial_weights)
    del base
    tf.keras.backend.clear_session()
    gc.collect()

    seed_everything(training_seed)
    global_model = build_diabetes_model(data["input_dim"], LEARNING_RATE)
    global_model.set_weights([np.array(w, copy=True) for w in initial_weights])
    param_bytes = model_parameter_bytes(global_model)

    natural_map = {
        cid: natural_steps(len(data["client_arrays"][cid][1]), BATCH_SIZE, LOCAL_EPOCHS)
        for cid in selected_clients
    }

    start = time.perf_counter()
    total_steps = 0
    comm_raw_bytes = 0

    for round_idx in range(1, NUM_ROUNDS_FL + 1):
        global_weights = [np.array(w, copy=True) for w in global_model.get_weights()]
        local_weights = []
        local_sizes = []

        for j, cid in enumerate(selected_clients):
            Xc, yc = data["client_arrays"][cid]
            local_model = build_diabetes_model(data["input_dim"], LEARNING_RATE)
            local_model.set_weights(global_weights)
            train_exact_steps(
                local_model, Xc, yc,
                steps=int(natural_map[cid]),
                batch_size=BATCH_SIZE,
                seed=int(training_seed + 1000 * round_idx + 17 * j),
                class_weights=data["class_weights"],
            )
            local_weights.append([np.array(w, copy=True) for w in local_model.get_weights()])
            local_sizes.append(len(yc))
            total_steps += int(natural_map[cid])
            del local_model
            tf.keras.backend.clear_session()

        agg = aggregate_weights(local_weights, local_sizes)
        global_model = build_diabetes_model(data["input_dim"], LEARNING_RATE)
        global_model.set_weights(agg)
        comm_raw_bytes += len(selected_clients) * 2 * param_bytes

    runtime_s = float(time.perf_counter() - start)
    metrics = evaluate_model(global_model, data["X_test"], data["y_test"])
    communication_mb = float((comm_raw_bytes * 1.12) / (1024 ** 2))
    energy_wh = float(ESTIMATED_POWER_W * runtime_s / 3600.0)

    del global_model
    tf.keras.backend.clear_session()
    gc.collect()

    return {
        **metrics,
        "runtime_s": runtime_s,
        "communication_mb": communication_mb,
        "energy_wh": energy_wh,
        "optimizer_steps": int(total_steps),
        "participants": int(len(selected_clients)),
        "selected_client_ids": ";".join(selected_clients),
        "initial_weights_sha256": initial_hash,
    }


def train_unique_cohorts(
    cohort_mapping: dict,
    data: dict,
    experiment_root: Path,
) -> pd.DataFrame:
    checkpoint_csv = experiment_root / "unique_cohort_performance_checkpoint.csv"
    checkpoint_state = experiment_root / "checkpoint_state.json"

    # cohort_mapping maps cohort_key -> cohort_id.
    jobs = []
    for cohort_key, cohort_id in cohort_mapping.items():
        clients = [x for x in str(cohort_key).split(";") if x]
        for seed in TRAINING_RUN_SEEDS:
            jobs.append((cohort_id, cohort_key, clients, seed))

    completed = set(load_checkpoint_state(checkpoint_state).get("completed", []))

    for job_idx, (cohort_id, cohort_key, clients, seed) in enumerate(jobs, start=1):
        key = f"{cohort_id}|seed={seed}"
        if key in completed:
            print(f"CHECKPOINT FOUND -> SKIP {key}")
            continue

        print_banner(
            f"EXPERIMENT E TRAINING {job_idx}/{len(jobs)} | {cohort_id} | seed={seed}"
        )
        print(f"Admitted clients ({len(clients)}): {clients}")

        result = train_tadp_vr_cohort(clients, data, seed)
        row = {
            "cohort_id": cohort_id,
            "cohort_key": cohort_key,
            "seed": int(seed),
            **result,
        }
        upsert_csv(row, checkpoint_csv, key_cols=["cohort_id", "seed"])
        mark_checkpoint_complete(checkpoint_state, key)
        completed.add(key)

        print(
            f"RESULT | Acc={result['accuracy']:.4f} | F1={result['f1_macro']:.4f} | "
            f"AUC={result['roc_auc_ovr_macro']:.4f} | steps={result['optimizer_steps']}"
        )

    perf = pd.read_csv(checkpoint_csv)
    perf = perf.sort_values(["cohort_id", "seed"]).reset_index(drop=True)
    return perf


# ======================================================================================
# 9. UTILITY DELTAS VS REFERENCE TADP-VR
# ======================================================================================

def summarize_cohort_performance(perf: pd.DataFrame) -> pd.DataFrame:
    metrics = [
        "accuracy", "precision_macro", "recall_macro", "f1_macro",
        "roc_auc_ovr_macro", "runtime_s", "communication_mb", "energy_wh",
        "optimizer_steps", "participants",
    ]
    rows = []
    for cohort_id, d in perf.groupby("cohort_id", sort=False):
        row = {
            "cohort_id": cohort_id,
            "cohort_key": str(d["cohort_key"].iloc[0]),
            "n_runs": int(len(d)),
        }
        for m in metrics:
            vals = pd.to_numeric(d[m], errors="coerce").to_numpy(float)
            vals = vals[np.isfinite(vals)]
            row[f"{m}_mean"] = float(np.mean(vals)) if len(vals) else np.nan
            row[f"{m}_sd"] = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
            lo, hi = ci95_mean(vals)
            row[f"{m}_ci95_low"] = lo
            row[f"{m}_ci95_high"] = hi
        rows.append(row)
    return pd.DataFrame(rows)


def build_policy_utility_impact(
    governance_summary: pd.DataFrame,
    perf: pd.DataFrame,
):
    reference = perf.loc[perf["cohort_id"].eq("REFERENCE_COHORT")].copy()
    if sorted(reference["seed"].astype(int).tolist()) != TRAINING_RUN_SEEDS:
        raise RuntimeError("Reference cohort does not contain all five training seeds.")

    paired_rows = []
    summary_rows = []

    for cfg in governance_summary.itertuples(index=False):
        base = {c: getattr(cfg, c) for c in governance_summary.columns}

        if not bool(cfg.valid_policy) or not str(cfg.cohort_id):
            summary_rows.append({
                **base,
                "utility_available": False,
                "delta_accuracy_mean": np.nan,
                "delta_f1_macro_mean": np.nan,
                "delta_roc_auc_ovr_macro_mean": np.nan,
            })
            continue

        current = perf.loc[perf["cohort_id"].eq(str(cfg.cohort_id))].copy()
        merged = current.merge(
            reference,
            on="seed",
            suffixes=("_current", "_reference"),
            validate="one_to_one",
        )

        for _, r in merged.iterrows():
            paired_rows.append({
                "configuration": str(cfg.configuration),
                "family": str(cfg.family),
                "cohort_id": str(cfg.cohort_id),
                "seed": int(r["seed"]),
                "accuracy_current": float(r["accuracy_current"]),
                "accuracy_reference": float(r["accuracy_reference"]),
                "delta_accuracy": float(r["accuracy_current"] - r["accuracy_reference"]),
                "f1_macro_current": float(r["f1_macro_current"]),
                "f1_macro_reference": float(r["f1_macro_reference"]),
                "delta_f1_macro": float(r["f1_macro_current"] - r["f1_macro_reference"]),
                "roc_auc_ovr_macro_current": float(r["roc_auc_ovr_macro_current"]),
                "roc_auc_ovr_macro_reference": float(r["roc_auc_ovr_macro_reference"]),
                "delta_roc_auc_ovr_macro": float(
                    r["roc_auc_ovr_macro_current"] - r["roc_auc_ovr_macro_reference"]
                ),
            })

        row = {**base, "utility_available": True}
        for metric in ["accuracy", "f1_macro", "roc_auc_ovr_macro"]:
            cur = merged[f"{metric}_current"].to_numpy(float)
            ref = merged[f"{metric}_reference"].to_numpy(float)
            delta = cur - ref

            row[f"{metric}_mean"] = float(np.mean(cur))
            row[f"{metric}_sd"] = float(np.std(cur, ddof=1)) if len(cur) > 1 else 0.0
            row[f"reference_{metric}_mean"] = float(np.mean(ref))
            row[f"delta_{metric}_mean"] = float(np.mean(delta))
            row[f"delta_{metric}_sd"] = float(np.std(delta, ddof=1)) if len(delta) > 1 else 0.0
            lo, hi = ci95_mean(delta)
            row[f"delta_{metric}_ci95_low"] = lo
            row[f"delta_{metric}_ci95_high"] = hi

        summary_rows.append(row)

    return pd.DataFrame(summary_rows), pd.DataFrame(paired_rows)


def build_lodo_sensitivity_summary(policy_impact: pd.DataFrame) -> pd.DataFrame:
    x = policy_impact.loc[
        policy_impact["family"].eq("LODO") & policy_impact["valid_policy"].eq(True)
    ].copy()
    if x.empty:
        return x

    # Descriptive sensitivity score only — NOT a normative importance ranking.
    for m in ["delta_accuracy_mean", "delta_f1_macro_mean", "delta_roc_auc_ovr_macro_mean"]:
        x[f"abs_{m}"] = x[m].abs()
    x["mean_absolute_utility_delta"] = x[[
        "abs_delta_accuracy_mean", "abs_delta_f1_macro_mean",
        "abs_delta_roc_auc_ovr_macro_mean"
    ]].mean(axis=1)

    return x.sort_values(
        ["decision_flip_count", "mean_absolute_utility_delta"],
        ascending=[False, False],
    ).reset_index(drop=True)


# ======================================================================================
# 10. FIGURES — SIMPLE, MANUSCRIPT-FRIENDLY
# ======================================================================================

def make_figures(policy_impact: pd.DataFrame, output_dir: Path):
    import matplotlib.pyplot as plt

    changed = policy_impact.loc[
        policy_impact["valid_policy"].eq(True)
        & policy_impact["configuration"].ne("REFERENCE")
        & (
            policy_impact["decision_flip_count"].fillna(0).gt(0)
            | policy_impact["delta_roc_auc_ovr_macro_mean"].fillna(0).abs().gt(1e-12)
        )
    ].copy()

    if not changed.empty:
        changed = changed.sort_values(
            ["family", "perturbation_pct", "configuration"]
        ).reset_index(drop=True)

        # Figure E1: admissions gained/lost.
        fig, ax = plt.subplots(figsize=(max(10, 0.48 * len(changed)), 5.5))
        x = np.arange(len(changed))
        ax.bar(x, changed["gained_count"].to_numpy(float), label="Gained")
        ax.bar(x, -changed["lost_count"].to_numpy(float), label="Lost")
        ax.axhline(0, linewidth=1)
        ax.set_ylabel("Clients gained / lost vs reference")
        ax.set_xticks(x)
        ax.set_xticklabels(changed["configuration"], rotation=70, ha="right")
        ax.legend()
        fig.tight_layout()
        fig.savefig(output_dir / "Figure_E1_Admission_Gain_Loss.png", dpi=300, bbox_inches="tight")
        plt.close(fig)

        # Figure E2: macro-AUC delta.
        fig, ax = plt.subplots(figsize=(max(10, 0.48 * len(changed)), 5.5))
        ax.bar(x, changed["delta_roc_auc_ovr_macro_mean"].to_numpy(float))
        ax.axhline(0, linewidth=1)
        ax.set_ylabel("Mean paired Macro AUC delta vs reference")
        ax.set_xticks(x)
        ax.set_xticklabels(changed["configuration"], rotation=70, ha="right")
        fig.tight_layout()
        fig.savefig(output_dir / "Figure_E2_Utility_Delta_MacroAUC.png", dpi=300, bbox_inches="tight")
        plt.close(fig)

    lodo = policy_impact.loc[policy_impact["family"].eq("LODO")].copy()
    if not lodo.empty:
        lodo = lodo.sort_values("removed_dimension")
        fig, ax = plt.subplots(figsize=(8.5, 5.0))
        x = np.arange(len(lodo))
        ax.bar(x, lodo["delta_roc_auc_ovr_macro_mean"].to_numpy(float))
        ax.axhline(0, linewidth=1)
        ax.set_ylabel("Mean paired Macro AUC delta vs reference")
        ax.set_xticks(x)
        ax.set_xticklabels(lodo["removed_dimension"])
        ax.set_xlabel("Removed TADP dimension")
        fig.tight_layout()
        fig.savefig(output_dir / "Figure_E3_LODO_Utility_Delta.png", dpi=300, bbox_inches="tight")
        plt.close(fig)


# ======================================================================================
# 11. PACKAGE RESULTS
# ======================================================================================

def package_results(output_dir: Path) -> Path:
    manifest = []
    for p in sorted(output_dir.rglob("*")):
        if p.is_file():
            manifest.append({
                "relative_path": str(p.relative_to(output_dir)),
                "bytes": int(p.stat().st_size),
                "sha256": sha256_file(p),
            })
    pd.DataFrame(manifest).to_csv(output_dir / "RESULTS_FILE_MANIFEST.csv", index=False)

    zip_path = output_dir.parent / f"{output_dir.name}_RESULTS.zip"
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        for p in sorted(output_dir.rglob("*")):
            if p.is_file():
                zf.write(p, arcname=str(p.relative_to(output_dir)))
    return zip_path


# ======================================================================================
# 12. MAIN RUNNER
# ======================================================================================

def run_experiment_e():
    print_banner("TADP EXPERIMENT E v20.2 — FRESH-REFERENCE POLICY SENSITIVITY + FULL-POLICY LODO")
    print("Reference comparison: NEW TADP-VR runs generated inside Experiment E.")
    print("No Experiment A1 result ZIP or previous predictive metrics are loaded.")

    diabetes_csv = locate_diabetes_csv()
    print(f"Diabetes CSV: {diabetes_csv}")

    experiment_root = choose_experiment_root(
        "TADP_EXPERIMENT_E_" + EXPERIMENT_VERSION,
        use_drive=USE_GOOGLE_DRIVE_CHECKPOINTS,
    )

    # 1) Recreate the K=10 data preparation and TRAIN-only DQ evidence fresh.
    data = prepare_diabetes_for_experiment_e(diabetes_csv)
    data["partition_audit"].to_csv(experiment_root / "client_partition_audit.csv", index=False)
    data["leakage_audit"].to_csv(experiment_root / "leakage_audit.csv", index=False)
    data["dq_audit"].to_csv(experiment_root / "dq_train_only_audit.csv", index=False)
    data["dataset_metadata"].to_csv(experiment_root / "dataset_metadata.csv", index=False)

    # 2) Recreate the frozen controlled documentary evidence and final policy fresh.
    policy = healthcare_policy_table()
    policy.to_csv(experiment_root / "healthcare_admission_policy.csv", index=False)
    documentary_evidence, documentary_df = generate_controlled_documentary_evidence(
        data["client_ids"], FROZEN_EVIDENCE_ASSIGNMENT_SEED
    )
    documentary_df.to_csv(
        experiment_root / "controlled_documentary_evidence_matrix.csv", index=False
    )
    evidence_assignment_summary(documentary_df).to_csv(
        experiment_root / "evidence_assignment_summary.csv", index=False
    )

    factor_table, dim_scores = build_full_factor_table(
        data["client_ids"], documentary_evidence, data["dq_scores"], policy
    )
    factor_table.to_csv(experiment_root / "all_28_factor_evidence_by_client.csv", index=False)

    # 3) Fresh unmodified reference governance. This is the reference for ALL comparisons.
    reference_gov, _ = evaluate_policy_v20_2(
        factor_table=factor_table,
        dim_scores=dim_scores,
        hps_weights=HPS_WEIGHTS,
        lower_cut=REFERENCE_LOWER_CUT,
        upper_cut=REFERENCE_UPPER_CUT,
        review_cut=REFERENCE_REVIEW_CUT,
        dimension_floor=REFERENCE_DIMENSION_FLOOR,
        removed_dim=None,
    )
    reference_gov = assert_fresh_reference_design(reference_gov)
    reference_gov.to_csv(experiment_root / "fresh_reference_governance.csv", index=False)

    # 4) Build all sensitivity + LODO governance conditions relative to that fresh reference.
    configs = build_sensitivity_configurations()
    configs.to_csv(experiment_root / "policy_sensitivity_design.csv", index=False)
    gov_summary, client_level, _ = evaluate_all_configurations(
        configs, factor_table, dim_scores, HPS_WEIGHTS
    )
    gov_summary, cohort_mapping = assign_cohort_ids(gov_summary)
    gov_summary.to_csv(experiment_root / "governance_impact_summary.csv", index=False)
    client_level.to_csv(
        experiment_root / "governance_client_level_all_valid_configs.csv", index=False
    )

    cohort_map_df = pd.DataFrame([
        {
            "cohort_key": key,
            "cohort_id": cid,
            "selected_client_count": len([x for x in key.split(";") if x]),
            "selected_clients": key,
        }
        for key, cid in cohort_mapping.items()
    ]).sort_values("cohort_id")
    cohort_map_df.to_csv(experiment_root / "unique_admitted_cohorts.csv", index=False)

    print_banner("GOVERNANCE SENSITIVITY — ADMISSION GAIN / LOSS VS FRESH REFERENCE")
    show_cols = [
        "configuration", "family", "accepted_count", "gained_clients", "lost_clients",
        "net_admission_change", "decision_flip_count", "accepted_set_jaccard_vs_reference",
        "cohort_id",
    ]
    print(gov_summary.loc[gov_summary["valid_policy"].eq(True), show_cols].to_string(index=False))
    print(f"\nUnique admitted cohorts requiring fresh training: {len(cohort_mapping)}")

    # 5) Save fresh initialization hashes. No comparison to prior experiments.
    init_rows = []
    for seed in TRAINING_RUN_SEEDS:
        seed_everything(seed)
        m = build_diabetes_model(data["input_dim"], LEARNING_RATE)
        init_rows.append({
            "seed": seed,
            "initial_weights_sha256": sha256_weights(m.get_weights()),
        })
        del m
        tf.keras.backend.clear_session()
    pd.DataFrame(init_rows).to_csv(experiment_root / "initialization_audit.csv", index=False)

    # 6) Train every UNIQUE cohort fresh. REFERENCE_COHORT is included here and therefore
    #    receives five new TADP-VR runs in this Experiment-E execution.
    perf = train_unique_cohorts(cohort_mapping, data, experiment_root)
    perf.to_csv(experiment_root / "fresh_unique_cohort_performance.csv", index=False)
    summarize_cohort_performance(perf).to_csv(
        experiment_root / "fresh_unique_cohort_performance_summary_mean_sd_ci95.csv", index=False
    )

    fresh_reference_metrics = perf.loc[perf["cohort_id"].eq("REFERENCE_COHORT")].copy()
    if len(fresh_reference_metrics) != len(TRAINING_RUN_SEEDS):
        raise RuntimeError("Fresh reference TADP-VR did not complete all five seeds.")
    fresh_reference_metrics.to_csv(
        experiment_root / "fresh_reference_TADP_VR_metrics_5seed.csv", index=False
    )

    # 7) Admission + paired predictive utility deltas against the NEW reference runs.
    policy_impact, paired = build_policy_utility_impact(gov_summary, perf)
    policy_impact.to_csv(
        experiment_root / "policy_admission_utility_impact_vs_fresh_reference.csv", index=False
    )
    paired.to_csv(
        experiment_root / "policy_utility_delta_paired_by_seed_vs_fresh_reference.csv", index=False
    )

    lodo_summary = build_lodo_sensitivity_summary(policy_impact)
    lodo_summary.to_csv(experiment_root / "LODO_dimension_sensitivity_summary.csv", index=False)

    design_out = {
        "experiment_version": EXPERIMENT_VERSION,
        "comparison_reference": "fresh_TADP_VR_reference_trained_inside_Experiment_E",
        "previous_experiment_results_loaded": False,
        "diabetes_csv_sha256": sha256_file(diabetes_csv),
        "K": K_SUBMISSIONS,
        "training_seeds": TRAINING_RUN_SEEDS,
        "fl_rounds": NUM_ROUNDS_FL,
        "local_epochs": LOCAL_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "dirichlet_alpha": DIRICHLET_ALPHA,
        "global_split_seed": GLOBAL_SPLIT_SEED,
        "client_partition_seed": CLIENT_PARTITION_SEED,
        "frozen_evidence_assignment_seed": FROZEN_EVIDENCE_ASSIGNMENT_SEED,
        "reference_lower_cut": REFERENCE_LOWER_CUT,
        "reference_upper_cut": REFERENCE_UPPER_CUT,
        "reference_review_cut": REFERENCE_REVIEW_CUT,
        "reference_dimension_floor": REFERENCE_DIMENSION_FLOOR,
        "perturbation_percentages": PERTURBATION_PCTS,
        "hps_weights": HPS_WEIGHTS,
        "expected_reference_cohort": EXPECTED_REFERENCE_COHORT,
        "mandatory_minima_perturbed": False,
        "utility_primary_endpoint": "macro ROC-AUC",
        "training_efficiency": "fresh training once per unique admitted cohort and seed",
    }
    atomic_write_json(design_out, experiment_root / "policy_and_experiment_design.json")

    make_figures(policy_impact, experiment_root)

    print_banner("FINAL EXPERIMENT-E IMPACT VS FRESH REFERENCE")
    final_cols = [
        "configuration", "family", "accepted_count", "gained_clients", "lost_clients",
        "net_admission_change", "accepted_set_jaccard_vs_reference",
        "delta_accuracy_mean", "delta_f1_macro_mean", "delta_roc_auc_ovr_macro_mean",
    ]
    print(
        policy_impact.loc[policy_impact["valid_policy"].eq(True), final_cols]
        .to_string(index=False)
    )

    print_banner("FULL-POLICY LODO DIMENSION SENSITIVITY")
    if not lodo_summary.empty:
        lodo_cols = [
            "removed_dimension", "accepted_count", "gained_clients", "lost_clients",
            "decision_flip_count", "delta_accuracy_mean", "delta_f1_macro_mean",
            "delta_roc_auc_ovr_macro_mean", "mean_absolute_utility_delta",
        ]
        print(lodo_summary[lodo_cols].to_string(index=False))

    zip_path = package_results(experiment_root)
    print_banner("EXPERIMENT E COMPLETE")
    print(f"Results directory: {experiment_root}")
    print(f"Results ZIP: {zip_path}")
    try:
        from google.colab import files as colab_files
        colab_files.download(str(zip_path))
    except Exception:
        pass

    return {
        "experiment_root": experiment_root,
        "zip_path": zip_path,
        "fresh_reference_metrics": fresh_reference_metrics,
        "governance_summary": gov_summary,
        "policy_impact": policy_impact,
        "lodo_summary": lodo_summary,
        "performance": perf,
    }


# ======================================================================================
# RUN
# ======================================================================================

if __name__ == "__main__":
    RESULTS = run_experiment_e()



TADP EXPERIMENT E v20.2 — FRESH-REFERENCE POLICY SENSITIVITY + FULL-POLICY LODO
Reference comparison: NEW TADP-VR runs generated inside Experiment E.
No Experiment A1 result ZIP or previous predictive metrics are loaded.
Diabetes CSV: /content/diabetes_130US.csv
Google Drive checkpoint mount unavailable: mount failed
Local checkpoint root: /content/TADP_EXPERIMENT_E_TADP-E-v20.2-K10-FRESHREF-POLICY-SENSITIVITY-LODO-MANDATORYGATE-REVIEWSCORE325-5SEED-4ROUND
Fresh reference governance self-check: PASS
Fresh reference admitted cohort: ['A', 'B', 'G', 'H', 'I', 'J']

GOVERNANCE SENSITIVITY — ADMISSION GAIN / LOSS VS FRESH REFERENCE
          configuration           family  accepted_count gained_clients lost_clients  net_admission_change  decision_flip_count  accepted_set_jaccard_vs_reference        cohort_id
              REFERENCE        REFERENCE             6.0                                               0.0                  0.0                           1.000000 REFERENCE_COHORT
   

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>